# DeepShield V5
## Google Colab Training

Purpose:
Train the DeepShield V5 EfficientNetB2 model using the existing FAST10 face-cropped dataset.

Status:
Preparation only -- training will be started manually.

TRAINING HAS NOT STARTED YET.

In [ ]:
# CELL 2 -- DEPENDENCY CHECK
import importlib

required_packages = [
    ("TensorFlow", "tensorflow"),
    ("NumPy", "numpy"),
    ("Pandas", "pandas"),
    ("Scikit-learn", "sklearn"),
    ("Matplotlib", "matplotlib"),
    ("Pillow", "PIL")
]

print("=" * 50)
print("DEEPSHIELD V5 DEPENDENCY CHECK")
print("=" * 50)

all_pass = True
for display_name, module_name in required_packages:
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "installed")
        print(f"{display_name}: {version} -- PASS")
    except ImportError:
        print(f"{display_name}: NOT FOUND -- FAIL")
        all_pass = False

print("\nDEPENDENCY CHECK: COMPLETE")
print("=" * 50)


In [ ]:
# CELL 3 -- MOUNT GOOGLE DRIVE
import os
from pathlib import Path

mount_success = False
mydrive_accessible = False

try:
    from google.colab import drive
    drive.mount('/content/drive')
    mount_success = True
except Exception as e:
    print(f"[!] Error during drive.mount: {e}")

mydrive_path = Path("/content/drive/MyDrive")
if mydrive_path.exists():
    mydrive_accessible = True

print("=" * 50)
print("GOOGLE DRIVE CHECK")
print("=" * 50)
print(f"Drive mount: {'PASS' if mount_success else 'FAIL'}")
print(f"MyDrive accessible: {'PASS' if mydrive_accessible else 'FAIL'}")

if mount_success and mydrive_accessible:
    print("\nGOOGLE DRIVE: READY")
else:
    print("\nGOOGLE DRIVE: FAILED")
print("=" * 50)


In [ ]:
# CELL 4 -- CREATE TRAINING DIRECTORY
from pathlib import Path

target_dir = Path("/content/training")
target_dir.mkdir(parents=True, exist_ok=True)

dir_exists = target_dir.is_dir()

print("=" * 50)
print("COLAB TRAINING DIRECTORY CHECK")
print("=" * 50)
print("Directory:")
print(f" {target_dir}/")
print("\nStatus:")
print(f" {'PASS' if dir_exists else 'FAIL'}")

if dir_exists:
    print("\nCOLAB TRAINING DIRECTORY: READY")
else:
    print("\nCOLAB TRAINING DIRECTORY: FAILED")
print("=" * 50)


In [ ]:
# CELL 5 -- UPLOAD TRAINING SCRIPT
from pathlib import Path
import shutil
from google.colab import files

# Create /content/training/ if necessary
target_dir = Path("/content/training")
target_dir.mkdir(parents=True, exist_ok=True)
target_script = target_dir / "train_deepshield_v5.py"

if not target_script.exists():
    print("[*] Please select and upload 'train_deepshield_v5.py' from your computer:")
    print("    Local source: C:\\Users\\ADMIN\\Desktop\\DeepShield_V5_Final\\training\\train_deepshield_v5.py\n")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith("train_deepshield_v5.py") or fname.endswith(".py"):
            shutil.move(fname, str(target_script))
            break

script_exists = target_script.is_file()
file_size = target_script.stat().st_size if script_exists else 0

print("=" * 50)
print("TRAINING SCRIPT CHECK")
print("=" * 50)
print("Training script:")
print(f" {target_script}")
if script_exists:
    print(f"Size: {file_size:,} bytes")
print("\nStatus:")
print(f" {'PASS' if script_exists else 'FAIL'}")

if script_exists:
    print("\nTRAINING SCRIPT: READY")
else:
    print("\nTRAINING SCRIPT: FAILED")
print("=" * 50)


In [ ]:
# CELL 6 -- VERIFY DATASET
from pathlib import Path

dataset_root = Path("/content/drive/MyDrive/DeepShield_Datasets/Face_Cropped_FAST10")
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

expected_counts = {
    "train": 17208,
    "validation": 3688,
    "test": 3693
}
expected_total = sum(expected_counts.values())

actual_counts = {}
dataset_split_results = {}
mismatches = []

for split, exp_count in expected_counts.items():
    split_dir = dataset_root / split
    if split_dir.is_dir():
        count = len([f for f in split_dir.rglob("*") if f.is_file() and f.suffix.lower() in valid_exts])
    else:
        count = 0
    actual_counts[split] = count
    is_match = (count == exp_count)
    dataset_split_results[split] = is_match
    if not is_match:
        mismatches.append(split)

actual_total = sum(actual_counts.values())

print("=" * 50)
print("DEEPSHIELD V5 DATASET CHECK")
print("=" * 50)
print(f"Dataset:\n {dataset_root}\n")
print(f"Train:\n {actual_counts['train']:,} / {expected_counts['train']:,}\n")
print(f"Validation:\n {actual_counts['validation']:,} / {expected_counts['validation']:,}\n")
print(f"Test:\n {actual_counts['test']:,} / {expected_counts['test']:,}\n")
print(f"Total:\n {actual_total:,} / {expected_total:,}")
print("=" * 50)

if len(mismatches) == 0 and dataset_root.is_dir():
    print("\nDATASET: READY")
else:
    print("\nDATASET: CHECK REQUIRED")
    if not dataset_root.is_dir():
        print(f"[!] Dataset root folder not found at: {dataset_root}")
    for m in mismatches:
        print(f"[!] Count mismatch in '{m}' split: Actual {actual_counts[m]:,} vs Expected {expected_counts[m]:,}")
print("=" * 50)


In [ ]:
# CELL 7 -- VERIFY MANIFEST
from pathlib import Path
import pandas as pd

manifest_path = Path("/content/drive/MyDrive/DeepShield_Datasets/Metadata/face_crop_FAST10_manifest_split.csv")

expected_records = 24589
expected_split_counts = {
    "train": 17208,
    "validation": 3688,
    "test": 3693
}
expected_cols = [
    "frame_id", "filepath", "filename", "label", "label_name",
    "source_dataset", "video_group", "split", "face_path",
    "face_detected", "face_width", "face_height", "face_confidence"
]

print("=" * 50)
print("DEEPSHIELD V5 MANIFEST CHECK")
print("=" * 50)
print(f"Manifest:\n {manifest_path}\n")

manifest_ok = False
if manifest_path.is_file():
    df = pd.read_csv(manifest_path)
    total_records = len(df)
    print(f"Records:\n {total_records:,} (Expected: {expected_records:,})\n")

    print("Class Mapping:")
    print(" 0 = Real")
    print(" 1 = Fake\n")

    print("Important Columns Check:")
    present_cols = set(df.columns)
    for col in expected_cols:
        status = "PASS" if col in present_cols else "MISSING"
        print(f"  - {col}: {status}")

    print("\nActual Split Distribution (from 'split' column):")
    split_counts = df["split"].value_counts().to_dict() if "split" in df.columns else {}
    for s, exp in expected_split_counts.items():
        act = split_counts.get(s, 0)
        print(f"  {s.capitalize()}: {act:,} (Expected: {exp:,})")

    print("\nActual Label Distribution (from 'label' column):")
    if "label" in df.columns:
        label_dist = df["label"].value_counts().to_dict()
        for lbl, count in label_dist.items():
            lbl_str = "Real" if lbl == 0 else ("Fake" if lbl == 1 else "Unknown")
            print(f"  Label {lbl} ({lbl_str}): {count:,}")

    if "label_name" in df.columns:
        print("\nActual label_name Distribution:")
        for lname, count in df["label_name"].value_counts().items():
            print(f"  {lname}: {count:,}")

    # Validation checks
    records_match = (total_records == expected_records)
    splits_match = all(split_counts.get(s, 0) == exp for s, exp in expected_split_counts.items())
    labels_present = ("label" in df.columns) and (set(df["label"].unique()).issubset({0, 1}))

    if records_match and splits_match and labels_present:
        manifest_ok = True
        print("\nManifest: READY")
    else:
        print("\nManifest: CHECK REQUIRED")
else:
    print(f"[!] ERROR: Manifest file not found at: {manifest_path}")
    print("\nManifest: FAILED")

print("=" * 50)


In [ ]:
# CELL 8 -- VERIFY GPU
import tensorflow as tf

tf_version = tf.__version__
gpu_devices = tf.config.list_physical_devices('GPU')
gpu_count = len(gpu_devices)
gpu_ok = (gpu_count > 0)

device_names = []
for dev in gpu_devices:
    try:
        details = tf.config.experimental.get_device_details(dev)
        device_names.append(details.get('device_name', dev.name))
    except Exception:
        device_names.append(dev.name)

gpu_device_str = ", ".join(device_names) if device_names else "None"

print("=" * 50)
print("DEEPSHIELD V5 GPU CHECK")
print("=" * 50)
print(f"TensorFlow:\n {tf_version}\n")
print(f"GPU devices:\n {gpu_devices}\n")
print(f"GPU count:\n {gpu_count}\n")
print(f"GPU device:\n {gpu_device_str}")
print("=" * 50)

if gpu_ok:
    print("\nGPU STATUS: READY")
else:
    print("\nGPU STATUS: NOT AVAILABLE")
print("=" * 50)


In [ ]:
# CELL 9 -- SYNTAX CHECK
from pathlib import Path
import py_compile

target_script = Path("/content/training/train_deepshield_v5.py")

syntax_pass = False
print("=" * 50)
print("DEEPSHIELD V5 SYNTAX CHECK")
print("=" * 50)
print("File:")
print(f" {target_script}\n")

if target_script.is_file():
    try:
        py_compile.compile(str(target_script), doraise=True)
        syntax_pass = True
        print("Syntax:")
        print(" PASS\n")
        print("V5 TRAINING SCRIPT SYNTAX: PASS")
    except py_compile.PyCompileError as err:
        print("Syntax:")
        print(" FAIL\n")
        print(f"[!] Syntax Error details:\n{err}")
    except Exception as ex:
        print("Syntax:")
        print(" FAIL\n")
        print(f"[!] Error during compilation check:\n{ex}")
else:
    print("Syntax:")
    print(" FAIL\n")
    print(f"[!] File not found at: {target_script}")

print("=" * 50)


In [ ]:
# CELL 10 -- REVERSED VS ORIGINAL PREDICTION EVALUATION EXPERIMENT
# Purpose: Purely experimental evaluation testing reversed interpretation (LOW score = Fake, HIGH score = Real)
# Note: Model weights, class mapping, and training scripts are NOT permanently modified.
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Ensure Google Drive is accessible
if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

# Paths for manifest and dataset (local fast NVMe storage preferred, Drive fallback)
LOCAL_MANIFEST = Path("/content/DeepShield_FAST10_manifest_split.csv")
DRIVE_MANIFEST = Path("/content/drive/MyDrive/DeepShield_Datasets/Metadata/face_crop_FAST10_manifest_split.csv")
MANIFEST_PATH = LOCAL_MANIFEST if LOCAL_MANIFEST.is_file() else DRIVE_MANIFEST

LOCAL_DATASET = Path("/content/DeepShield_FAST10")
DRIVE_DATASET = Path("/content/drive/MyDrive/DeepShield_Datasets/Face_Cropped_FAST10")
DATASET_ROOT = LOCAL_DATASET if LOCAL_DATASET.is_dir() else DRIVE_DATASET

# Potential model candidate paths to evaluate
MODEL_CANDIDATES = [
    Path("/content/drive/MyDrive/DeepShield_Datasets/models/deepshield_final.keras"),
    Path("/content/drive/MyDrive/DeepShield_V5/deepshield_v5_best.keras"),
    Path("/content/drive/MyDrive/DeepShield_V5/checkpoints/deepshield_v5_stage2_best.keras"),
    Path("/content/drive/MyDrive/DeepShield_V5/checkpoints/deepshield_v5_stage1_best.keras"),
    Path("models/deepshield_final.keras"),
    Path("/content/models/deepshield_final.keras")
]

model_path = None
for cand in MODEL_CANDIDATES:
    if cand.is_file():
        model_path = cand
        break

print("=" * 60)
print("DEEPSHIELD V5: PREDICTION INTERPRETATION EXPERIMENT")
print("=" * 60)
print(f"Dataset root:  {DATASET_ROOT}")
print(f"Manifest:      {MANIFEST_PATH}")
print(f"Model path:    {model_path if model_path else 'No model file found'}")
print("=" * 60)

if model_path is None or not model_path.is_file():
    print("[!] Notice: No trained model file found at candidate paths.")
    print("    Available candidates searched:")
    for cand in MODEL_CANDIDATES:
        print(f"      - {cand}")
    print("[*] Place an existing model file at one of the paths above to run this evaluation.")
else:
    print(f"[*] Loading model from: {model_path}...")
    model = tf.keras.models.load_model(str(model_path))
    print("[+] Model loaded successfully.")

    # Load manifest and filter validation split
    df = pd.read_csv(MANIFEST_PATH)
    eval_df = df[df['split'] == 'validation'].copy()
    split_name = 'validation'
    if len(eval_df) == 0:
        eval_df = df[df['split'] == 'test'].copy()
        split_name = 'test'

    print(f"[*] Evaluating on '{split_name}' split ({len(eval_df):,} samples)...")

    # Resolve file paths and ground truth labels (0 = Real, 1 = Fake)
    split_dir = DATASET_ROOT / split_name
    file_paths = []
    y_true = []
    for _, row in eval_df.iterrows():
        fname = str(row['filename']) if 'filename' in row and pd.notna(row['filename']) else Path(str(row['filepath'])).name
        p = split_dir / fname
        if not p.is_file():
            p = DATASET_ROOT / str(row.get('filepath', '')).replace("\\", "/")
        if p.is_file():
            file_paths.append(str(p))
            y_true.append(int(row['label']))

    y_true = np.array(y_true)
    print(f"[+] Resolved {len(file_paths):,} / {len(eval_df):,} images.")

    # Build evaluation tf.data pipeline
    def parse_img(path):
        img_bytes = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img_bytes, channels=3)
        img = tf.image.resize(img, (224, 224))
        img = tf.cast(img, tf.float32)
        return img

    eval_ds = tf.data.Dataset.from_tensor_slices(file_paths)
    eval_ds = eval_ds.map(parse_img, num_parallel_calls=tf.data.AUTOTUNE)
    eval_ds = eval_ds.batch(32).prefetch(tf.data.AUTOTUNE)

    print("[*] Running inference...")
    raw_scores = model.predict(eval_ds, verbose=1).flatten()

    # A. ORIGINAL INTERPRETATION: HIGH score = Fake (1), LOW score = Real (0)
    y_pred_orig = (raw_scores >= 0.5).astype(int)
    acc_orig = accuracy_score(y_true, y_pred_orig)
    prec_orig = precision_score(y_true, y_pred_orig, zero_division=0)
    rec_orig = recall_score(y_true, y_pred_orig, zero_division=0)
    f1_orig = f1_score(y_true, y_pred_orig, zero_division=0)
    try:
        auc_orig = roc_auc_score(y_true, raw_scores)
    except Exception:
        auc_orig = float("nan")

    # B. REVERSED INTERPRETATION: LOW score = Fake (1), HIGH score = Real (0)
    y_pred_rev = (raw_scores < 0.5).astype(int)
    acc_rev = accuracy_score(y_true, y_pred_rev)
    prec_rev = precision_score(y_true, y_pred_rev, zero_division=0)
    rec_rev = recall_score(y_true, y_pred_rev, zero_division=0)
    f1_rev = f1_score(y_true, y_pred_rev, zero_division=0)
    try:
        # For reversed interpretation, ranking score for Fake is (1.0 - raw_scores)
        auc_rev = roc_auc_score(y_true, 1.0 - raw_scores)
    except Exception:
        auc_rev = float("nan")

    print("\n" + "=" * 60)
    print("ORIGINAL PREDICTION EVALUATION")
    print("=" * 60)
    print("Interpretation:")
    print("HIGH score -> FAKE")
    print("LOW score  -> REAL")
    print(f"Accuracy:  {acc_orig:.4f}")
    print(f"Precision: {prec_orig:.4f}")
    print(f"Recall:    {rec_orig:.4f}")
    print(f"F1:        {f1_orig:.4f}")
    print(f"ROC-AUC:   {auc_orig:.4f}")

    print("\n" + "=" * 60)
    print("REVERSED PREDICTION EVALUATION")
    print("=" * 60)
    print("Interpretation:")
    print("LOW score  -> FAKE")
    print("HIGH score -> REAL")
    print(f"Accuracy:  {acc_rev:.4f}")
    print(f"Precision: {prec_rev:.4f}")
    print(f"Recall:    {rec_rev:.4f}")
    print(f"F1:        {f1_rev:.4f}")
    print(f"ROC-AUC:   {auc_rev:.4f}")
    print("=" * 60)

    print("\nCOMPARISON SUMMARY:")
    print(f"Original  - Acc: {acc_orig:.4f} | Prec: {prec_orig:.4f} | Rec: {rec_orig:.4f} | F1: {f1_orig:.4f} | AUC: {auc_orig:.4f}")
    print(f"Reversed  - Acc: {acc_rev:.4f} | Prec: {prec_rev:.4f} | Rec: {rec_rev:.4f} | F1: {f1_rev:.4f} | AUC: {auc_rev:.4f}")
    if f1_rev > f1_orig and acc_rev > acc_orig:
        print(">> Evaluation Outcome: Reversed interpretation yielded higher metrics on this split.")
    elif f1_orig > f1_rev and acc_orig > acc_rev:
        print(">> Evaluation Outcome: Original interpretation yielded higher metrics on this split.")
    else:
        print(">> Evaluation Outcome: Metrics show mixed trade-offs between interpretations.")


In [ ]:
# CELL 11 -- SPEED OPTIMIZATION: LOCAL DATASET COPY & VERIFICATION
import os
import shutil
from pathlib import Path
import pandas as pd

# Auto-mount Google Drive if not already mounted in active runtime
if not Path("/content/drive/MyDrive").exists():
    print("[*] Google Drive is not mounted. Mounting now...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("[+] Google Drive successfully mounted.")
    except Exception as exc:
        print(f"[!] Drive auto-mount note: {exc}")

# Paths
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/DeepShield_Datasets/Face_Cropped_FAST10")
DRIVE_MANIFEST_PATH = Path("/content/drive/MyDrive/DeepShield_Datasets/Metadata/face_crop_FAST10_manifest_split.csv")
LOCAL_DATASET_ROOT = Path("/content/DeepShield_FAST10")
LOCAL_MANIFEST_PATH = Path("/content/DeepShield_FAST10_manifest_split.csv")

EXPECTED = {
    "train": 17208,
    "validation": 3688,
    "test": 3693,
    "total": 24589
}

print("=" * 60)
print("DEEPSHIELD V5: LOCAL DATASET SPEED OPTIMIZATION")
print("=" * 60)

# Helper to count image files in directory
def count_images(dir_path: Path) -> int:
    if not dir_path.is_dir():
        return 0
    valid_exts = {".jpg", ".jpeg", ".png"}
    return sum(1 for f in os.scandir(dir_path) if f.is_file() and Path(f.name).suffix.lower() in valid_exts)

# Step 1: Check if already copied with exact counts
train_cnt = count_images(LOCAL_DATASET_ROOT / "train")
val_cnt = count_images(LOCAL_DATASET_ROOT / "validation")
test_cnt = count_images(LOCAL_DATASET_ROOT / "test")
total_cnt = train_cnt + val_cnt + test_cnt
manifest_ready = LOCAL_MANIFEST_PATH.is_file()

if (total_cnt == EXPECTED["total"] and train_cnt == EXPECTED["train"] and 
    val_cnt == EXPECTED["validation"] and test_cnt == EXPECTED["test"] and manifest_ready):
    print("[+] Local dataset already exists with verified counts. Reusing local copy:")
    print(f"    Train: {train_cnt:,} | Validation: {val_cnt:,} | Test: {test_cnt:,} | Total: {total_cnt:,}")
else:
    # Verify Drive source exists first
    if not DRIVE_DATASET_ROOT.is_dir():
        raise FileNotFoundError(f"Source dataset not found on Google Drive: {DRIVE_DATASET_ROOT}")
    if not DRIVE_MANIFEST_PATH.is_file():
        raise FileNotFoundError(f"Source manifest not found on Google Drive: {DRIVE_MANIFEST_PATH}")

    # Copy manifest
    print(f"[*] Copying manifest to local storage: {LOCAL_MANIFEST_PATH}...")
    shutil.copy2(DRIVE_MANIFEST_PATH, LOCAL_MANIFEST_PATH)
    print("[+] Manifest copied successfully.")

    # Copy dataset using rsync (fastest, preserves timestamps, resumable) with fallback to cp -r
    LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"[*] Copying FAST10 dataset to high-speed NVMe storage: {LOCAL_DATASET_ROOT}...")
    rsync_cmd = f"rsync -a '{DRIVE_DATASET_ROOT}/' '{LOCAL_DATASET_ROOT}/'"
    ret = os.system(rsync_cmd)
    if ret != 0:
        print("[*] rsync returned non-zero, using cp -r fallback...")
        os.system(f"cp -r '{DRIVE_DATASET_ROOT}/.' '{LOCAL_DATASET_ROOT}/'")
    print("[+] Dataset copy complete.")

# Step 2: Verification of local files and manifest
print("\n" + "=" * 60)
print("VERIFYING LOCAL REPLICATED DATASET & MANIFEST")
print("=" * 60)

local_train = count_images(LOCAL_DATASET_ROOT / "train")
local_val = count_images(LOCAL_DATASET_ROOT / "validation")
local_test = count_images(LOCAL_DATASET_ROOT / "test")
local_total = local_train + local_val + local_test

df_manifest = pd.read_csv(LOCAL_MANIFEST_PATH)
manifest_total = len(df_manifest)

print(f"Local Dataset Path:  {LOCAL_DATASET_ROOT}")
print(f"Local Manifest Path: {LOCAL_MANIFEST_PATH}")
print(f"Train Images:        {local_train:,}  (Expected: {EXPECTED['train']:,})")
print(f"Validation Images:   {local_val:,}  (Expected: {EXPECTED['validation']:,})")
print(f"Test Images:         {local_test:,}  (Expected: {EXPECTED['test']:,})")
print(f"Total Images:        {local_total:,}  (Expected: {EXPECTED['total']:,})")
print(f"Manifest Records:    {manifest_total:,}  (Expected: {EXPECTED['total']:,})")
print("=" * 60)

assert local_train == EXPECTED["train"], f"Mismatch train count: {local_train}"
assert local_val == EXPECTED["validation"], f"Mismatch val count: {local_val}"
assert local_test == EXPECTED["test"], f"Mismatch test count: {local_test}"
assert local_total == EXPECTED["total"], f"Mismatch total count: {local_total}"
assert manifest_total == EXPECTED["total"], f"Mismatch manifest count: {manifest_total}"
print("\n" + "=" * 50)
print("LOCAL DATASET: READY")
print("LOCAL MANIFEST: READY")
print(f"TOTAL: {local_total:,}")
print("TRAINING: NOT STARTED")
print("=" * 50)


In [ ]:
# CELL 12 -- START DEEPSHIELD V5 TRAINING
# Execute the optimized two-stage training script in Google Colab
!python /content/training/train_deepshield_v5.py
